# The REG109_DESIGNATED_STATES table

Welcome to table **REG109_DESIGNATED_STATES** in PATSTAT Register. This table contains the information about states designated for an application. There are these types of designated states:
* Member states of the European Patent Convention (EPC)
* Extension states
* Validation states

In [1]:
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import REG109_DESIGNATED_STATES
from sqlalchemy import func
import pandas as pd

# Initialise the PATSTAT client
patstat = PatstatClient(env='PROD')

# Access ORM
db = patstat.orm()

## ID (Primary Key)

Technical identifier for an application, without business meaning. Its values will not change from one PATSTAT edition to the next.

In [2]:
i = db.query(
    REG109_DESIGNATED_STATES.id
).limit(1000)

df = patstat.df(i)
df

,id
0,24888565
1,18159170
2,97201825
3,23739610
4,13752014
...,...
995,9721585
996,14720031
997,15802000
998,3731376


## STATE_TYPE

This attribute indicates whether the designated state is a member state, an extension state or a validation state.

Let's count how many applications we have for each type.

In [3]:
kind = db.query(
    REG109_DESIGNATED_STATES.state_type,
    func.count(REG109_DESIGNATED_STATES.id).label('number_of_applications')
).group_by(
    REG109_DESIGNATED_STATES.state_type
).order_by(
    func.count(REG109_DESIGNATED_STATES.id)
)

kind_df = patstat.df(kind)
kind_df

,state_type,number_of_applications
0,VAL,2056911
1,EXT,3474165
2,MEM,8332870


## CHANGE_DATE

It is the date of when the record was saved in the database.

In [4]:
change_date = db.query(
    REG109_DESIGNATED_STATES.change_date,
    REG109_DESIGNATED_STATES.id
).limit(100)

change_date_df = patstat.df(change_date)
change_date_df

,change_date,id
0,9999-12-31,24888565
1,2018-08-03,18159170
2,1998-07-24,97201825
3,9999-12-31,23739610
4,9999-12-31,13752014
...,...,...
95,2005-03-11,4814815
96,2008-02-15,7012465
97,9999-12-31,15192035
98,2000-02-25,99300318


## BULLETIN_YEAR

For actions that have been published in the European Patent Bulletin, it is the year of the publication in the European Patent Bulletin. The default value is 0, used for applications that are not published or for which the year is not known. The format is YYYY otherwise.

In [5]:
years = db.query(
    REG109_DESIGNATED_STATES.bulletin_year,
    REG109_DESIGNATED_STATES.id
).limit(1000)

years_df = patstat.df(years)
years_df

,bulletin_year,id
0,0,24888565
1,2018,18159170
2,1998,97201825
3,0,23739610
4,0,13752014
...,...,...
995,0,9721585
996,0,14720031
997,2018,15802000
998,0,3731376


## BULLETIN_NR

This is the issue number of the EPO Bulletin for actions that have been published in it. This attribute indicates the calendar week the European Patent Bulletin was published. The default value 0 is used when the attribute `bulletin_year` is 0.

In [6]:
bulletin_nr = db.query(
    REG109_DESIGNATED_STATES.id,
    REG109_DESIGNATED_STATES.bulletin_nr,
    REG109_DESIGNATED_STATES.bulletin_year
).limit(100)

bulletin_nr_df = patstat.df(bulletin_nr)
bulletin_nr_df

,id,bulletin_nr,bulletin_year
0,24888565,0,0
1,18159170,36,2018
2,97201825,37,1998
3,23739610,0,0
4,13752014,0,0
...,...,...,...
95,4814815,0,0
96,7012465,12,2008
97,15192035,0,0
98,99300318,15,2000


## DESIGNATED_STATES

List of designated states. Let's display the applications with corresponding list of designated state and the state type.

In [7]:
des_states = db.query(
    REG109_DESIGNATED_STATES.id,
    REG109_DESIGNATED_STATES.state_type,
    REG109_DESIGNATED_STATES.designated_states
).limit(1000)

des_states_df = patstat.df(des_states)
des_states_df

,id,state_type,designated_states
0,13005000,EXT,"BA,ME"
1,8866236,MEM,"AT,BE,BG,CH,CY,CZ,DE,DK,EE,ES,FI,FR,GB,GR,HR,H..."
2,5749416,EXT,"AL,BA,HR,LV,MK,YU"
3,22854131,MEM,"AL,AT,BE,BG,CH,CY,CZ,DE,DK,EE,ES,FI,FR,GB,GR,H..."
4,11766540,MEM,"AL,AT,BE,BG,CH,CY,CZ,DE,DK,EE,ES,FI,FR,GB,GR,H..."
...,...,...,...
995,917693,MEM,"AT,BE,CH,CY,DE,DK,ES,FI,FR,GB,GR,IE,IT,LI,LU,M..."
996,14761319,MEM,"AL,AT,BE,BG,CH,CY,CZ,DE,DK,EE,ES,FI,FR,GB,GR,H..."
997,24907876,EXT,BA
998,15751408,EXT,"BA,ME"


We can also check which states appear more frequently as designated states.

In [8]:
# Create an empty list to add the number of symbols for each application id
num_labels = []

# Iterate over the list of symbols (over the rows under "ipc_text")
for label in des_states_df['designated_states']:
    # Convert label in a list: a new element is defined every time a comma is encountered
    sequence = label.split(",")
    # Append the length of list just created
    num_labels.append(len(sequence))

# Create a new column of the dataframes containing the length of the symbols sequence
des_states_df['designated_states'] = num_labels
# Sort the dataframe by the number_of_labels attribute in descending order
des_states_df = des_states_df.sort_values(by='designated_states', ascending=False)
# Save the top 10 applications
most_labels = des_states_df.head(100)
most_labels

,id,state_type,designated_states
155,23754456,MEM,39
474,24810048,MEM,39
189,23924831,MEM,39
733,24760307,MEM,39
770,23788070,MEM,39
...,...,...,...
313,19213204,MEM,38
312,12857160,MEM,38
316,15812945,MEM,38
595,22868011,MEM,38
